# Perplexity and Context Impact Evaluation

Comprehensive evaluation of language modeling quality and context benefit:
- Baseline perplexity on WikiText-103 and C4
- Context impact via continuation prediction
- Statistical analysis with confidence intervals
- Cross-dataset generalization

Models: FP16, GPTQ, AWQ, NF4 (base and instruct)

Setup and Installation

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

subprocess.run(['rm', '-rf', '/kaggle/working/qrag'], check=False)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/zahraamselim/qrag.git"
subprocess.run(['git', 'clone', repo_url, '/kaggle/working/qrag'], check=True)

sys.path.insert(0, '/kaggle/working/qrag')
os.chdir('/kaggle/working/qrag')

print("Repository cloned successfully")

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch',
    'transformers==4.51.3',
    'accelerate',
    'exllamav2',
    'autoawq',
    'bitsandbytes',
    'datasets',
    'scipy',
    'numpy',
    'huggingface-hub'
], check=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Dependencies installed")

Imports

In [ ]:
import gc
import torch
import logging
import numpy as np
from datasets import load_dataset
from scipy import stats

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

from utils.io import save_json
from utils.stats import bootstrap_ci, analyze_distribution
from metrics.perplexity import (
    compute_perplexity,
    compute_sequence_nll,
    compute_nll_conditional,
    bits_per_byte,
    perplexity_statistics
)

print("Imports successful")

Configuration

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/perplexity_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

logger.info(f"Output: {OUTPUT_DIR}")
logger.info(f"Device: {DEVICE}")

if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
DATASETS = [
    {
        'name': 'wikitext',
        'config': 'wikitext-103-v1',
        'split': 'test',
        'field': 'text'
    },
    {
        'name': 'allenai/c4',
        'config': 'en',
        'split': 'validation',
        'field': 'text',
        'streaming': True
    }
]

MAX_LENGTH = 2048
NUM_SAMPLES = 100
CONTEXT_SAMPLES = 50
MIN_DOC_LENGTH = 200

all_results = {}

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

Evaluation Functions

In [ ]:
def evaluate_perplexity(model, tokenizer, dataset, dataset_name, num_samples, 
                        max_length, backend='huggingface'):
    """Evaluate perplexity on dataset with robust sample selection."""
    logger.info(f"Evaluating {dataset_name} on {num_samples} samples")
    
    total_loss = 0.0
    total_tokens = 0
    sequence_ppls = []
    
    num_processed = 0
    num_attempted = 0
    max_attempts = num_samples * 100
    
    for item in dataset:
        if num_processed >= num_samples:
            break
        
        if num_attempted >= max_attempts:
            logger.warning(f"Reached max attempts ({max_attempts})")
            break
        
        num_attempted += 1
        
        text = item.get('text', '').strip()
        
        if len(text) < 100 or len(text.split()) < 20:
            continue
        
        alpha_ratio = sum(c.isalpha() for c in text) / len(text)
        if alpha_ratio < 0.5:
            continue
        
        nll, num_tokens = compute_sequence_nll(
            model,
            tokenizer,
            text,
            max_length=max_length,
            stride=512,
            use_sliding_window=False,
            backend=backend
        )
        
        if num_tokens == 0 or not np.isfinite(nll):
            continue
        
        total_loss += nll
        total_tokens += num_tokens
        
        seq_ppl = np.exp(nll / num_tokens)
        if np.isfinite(seq_ppl):
            sequence_ppls.append(seq_ppl)
        
        num_processed += 1
        
        if num_processed % 10 == 0:
            current_ppl = compute_perplexity(total_loss, total_tokens)
            logger.info(f"  Progress: {num_processed}/{num_samples} | Current PPL: {current_ppl:.2f}")
    
    if total_tokens == 0:
        return {'error': 'No valid tokens processed'}
    
    overall_ppl = compute_perplexity(total_loss, total_tokens)
    avg_loss = total_loss / total_tokens
    bpb = bits_per_byte(overall_ppl)
    stats_summary = perplexity_statistics(sequence_ppls)
    distribution = analyze_distribution(sequence_ppls)
    
    return {
        'perplexity': float(overall_ppl),
        'avg_loss': float(avg_loss),
        'bits_per_byte': float(bpb),
        'total_tokens': int(total_tokens),
        'num_sequences': int(num_processed),
        'sequence_statistics': stats_summary,
        'distribution_analysis': distribution
    }

def evaluate_context_impact(model, tokenizer, dataset, dataset_name, num_samples, 
                            min_doc_length, backend='huggingface'):
    """
    Evaluate context impact via continuation prediction.
    
    Splits documents in half and measures:
    - Perplexity of second half WITHOUT first half (cold start)
    - Perplexity of second half WITH first half (warm start)
    
    Reduction shows how much preceding context helps.
    """
    logger.info(f"Evaluating context impact on {dataset_name}")
    logger.info(f"Target: {num_samples} samples with min length {min_doc_length} words")
    
    ppls_without = []
    ppls_with = []
    
    num_processed = 0
    num_attempted = 0
    max_attempts = num_samples * 100
    
    for item in dataset:
        if num_processed >= num_samples:
            break
        
        if num_attempted >= max_attempts:
            logger.warning(f"Reached max attempts ({max_attempts})")
            break
        
        num_attempted += 1
        
        text = item.get('text', '').strip()
        
        if len(text.split()) < min_doc_length:
            continue
        
        alpha_ratio = sum(c.isalpha() for c in text) / len(text)
        if alpha_ratio < 0.5:
            continue
        
        try:
            if backend == 'exllama':
                tokens = tokenizer.encode(text, add_bos=True, encode_special_tokens=True)
                if tokens.dim() > 1:
                    tokens = tokens[0]
                token_list = tokens.tolist()
            else:
                encoding = tokenizer(text, return_tensors='pt', truncation=False)
                token_list = encoding['input_ids'][0].tolist()
            
            if len(token_list) < 100:
                continue
            
            mid_point = len(token_list) // 2
            
            context_tokens = token_list[:mid_point]
            continuation_tokens = token_list[mid_point:]
            
            if len(continuation_tokens) < 20:
                continue
            
            if backend == 'exllama':
                context_text = tokenizer.decode(torch.tensor(context_tokens))
                continuation_text = tokenizer.decode(torch.tensor(continuation_tokens))
            else:
                context_text = tokenizer.decode(context_tokens, skip_special_tokens=True)
                continuation_text = tokenizer.decode(continuation_tokens, skip_special_tokens=True)
            
            nll_without, tokens_without = compute_sequence_nll(
                model,
                tokenizer,
                continuation_text,
                max_length=2048,
                backend=backend
            )
            
            if tokens_without == 0 or not np.isfinite(nll_without):
                continue
            
            ppl_without = np.exp(nll_without / tokens_without)
            if not np.isfinite(ppl_without):
                continue
            
            nll_with, tokens_with = compute_nll_conditional(
                model,
                tokenizer,
                context_text,
                continuation_text,
                max_length=2048,
                backend=backend
            )
            
            if tokens_with == 0 or not np.isfinite(nll_with):
                continue
            
            ppl_with = np.exp(nll_with / tokens_with)
            if not np.isfinite(ppl_with):
                continue
            
            ppls_without.append(ppl_without)
            ppls_with.append(ppl_with)
            
            num_processed += 1
            
            if num_processed % 10 == 0:
                current_reduction = np.mean([(w - c) / w for w, c in zip(ppls_without, ppls_with) if w > 0])
                logger.info(f"  Progress: {num_processed}/{num_samples} | Current reduction: {current_reduction*100:.1f}%")
        
        except Exception as e:
            logger.debug(f"Context impact sample failed: {e}")
            continue
    
    if not ppls_with or not ppls_without:
        return {'error': 'No valid samples'}
    
    logger.info(f"Successfully processed {len(ppls_with)} samples")
    
    mean_without, lower_without, upper_without = bootstrap_ci(ppls_without)
    mean_with, lower_with, upper_with = bootstrap_ci(ppls_with)
    
    reductions = [(w - c) / w for w, c in zip(ppls_without, ppls_with) if w > 0]
    reduction_mean, reduction_lower, reduction_upper = bootstrap_ci(reductions)
    
    t_stat, p_value = stats.ttest_rel(ppls_without, ppls_with)
    
    effect_size = (np.mean(ppls_without) - np.mean(ppls_with)) / np.std(ppls_without)
    
    dist_without = analyze_distribution(ppls_without)
    dist_with = analyze_distribution(ppls_with)
    dist_reduction = analyze_distribution(reductions)
    
    return {
        'without_context': {
            'mean': mean_without,
            'ci': {'lower': lower_without, 'upper': upper_without},
            'median': float(np.median(ppls_without)),
            'std': float(np.std(ppls_without)),
            'distribution': dist_without
        },
        'with_context': {
            'mean': mean_with,
            'ci': {'lower': lower_with, 'upper': upper_with},
            'median': float(np.median(ppls_with)),
            'std': float(np.std(ppls_with)),
            'distribution': dist_with
        },
        'reduction': {
            'mean': reduction_mean,
            'ci': {'lower': reduction_lower, 'upper': reduction_upper},
            'percent': float(reduction_mean * 100),
            'median': float(np.median(reductions)),
            'distribution': dist_reduction
        },
        'significance': {
            't_statistic': float(t_stat),
            'p_value': float(p_value),
            'significant': bool(p_value < 0.05),
            'effect_size': float(effect_size)
        },
        'num_samples': len(ppls_with)
    }

Load Evaluation Datasets

In [ ]:
logger.info("Loading datasets")

loaded_datasets = {}

for ds_config in DATASETS:
    ds_name = ds_config['name']
    logger.info(f"Loading {ds_name}")
    
    try:
        if ds_config.get('streaming', False):
            if ds_config.get('config'):
                dataset = load_dataset(
                    ds_config['name'],
                    ds_config['config'],
                    split=ds_config['split'],
                    streaming=True,
                    trust_remote_code=True
                )
            else:
                dataset = load_dataset(
                    ds_config['name'],
                    split=ds_config['split'],
                    streaming=True,
                    trust_remote_code=True
                )
        else:
            if ds_config.get('config'):
                dataset = load_dataset(
                    ds_config['name'],
                    ds_config['config'],
                    split=ds_config['split'],
                    trust_remote_code=True
                )
            else:
                dataset = load_dataset(
                    ds_config['name'],
                    split=ds_config['split'],
                    trust_remote_code=True
                )
        
        loaded_datasets[ds_name] = dataset
        logger.info(f"  {ds_name} loaded")
    except Exception as e:
        logger.error(f"Failed to load {ds_name}: {e}")

logger.info(f"Loaded {len(loaded_datasets)} datasets")

FP16 Base

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

clear_memory()

logger.info("FP16 BASE MODEL")

tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-v0.1',
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()

fp16_base_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    fp16_base_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    fp16_base_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['fp16_base'] = {
    'results': fp16_base_results,
    'backend': 'huggingface',
    'quantization': 'none',
    'model_type': 'base'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'fp16_base_results.json')

FP16 Instruct

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

clear_memory()

logger.info("FP16 INSTRUCT MODEL")

tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2',
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()

fp16_instruct_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    fp16_instruct_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    fp16_instruct_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['fp16_instruct'] = {
    'results': fp16_instruct_results,
    'backend': 'huggingface',
    'quantization': 'none',
    'model_type': 'instruct'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'fp16_instruct_results.json')

GPTQ Base

In [ ]:
from exllamav2 import ExLlamaV2, ExLlamaV2Config, ExLlamaV2Cache, ExLlamaV2Tokenizer
from huggingface_hub import snapshot_download

clear_memory()

logger.info("GPTQ BASE MODEL")

model_dir = snapshot_download(
    'TheBloke/Mistral-7B-v0.1-GPTQ',
    allow_patterns=["*.json", "*.safetensors", "*.model"]
)

config = ExLlamaV2Config()
config.model_dir = model_dir
config.prepare()

model = ExLlamaV2(config)
cache = ExLlamaV2Cache(model, lazy=True)
model.load_autosplit(cache)
model.cache = cache

tokenizer = ExLlamaV2Tokenizer(config)

gptq_base_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='exllama')
    gptq_base_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='exllama'
    )
    gptq_base_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['gptq_base'] = {
    'results': gptq_base_results,
    'backend': 'exllamav2',
    'quantization': 'GPTQ',
    'model_type': 'base'
}

del model, cache, tokenizer, config
clear_memory()

save_json(all_results, OUTPUT_DIR / 'gptq_base_results.json')

GPTQ Instruct

In [ ]:
from exllamav2 import ExLlamaV2, ExLlamaV2Config, ExLlamaV2Cache, ExLlamaV2Tokenizer
from huggingface_hub import snapshot_download

clear_memory()

logger.info("GPTQ INSTRUCT MODEL")

model_dir = snapshot_download(
    'TheBloke/Mistral-7B-Instruct-v0.2-GPTQ',
    allow_patterns=["*.json", "*.safetensors", "*.model"]
)

config = ExLlamaV2Config()
config.model_dir = model_dir
config.prepare()

model = ExLlamaV2(config)
cache = ExLlamaV2Cache(model, lazy=True)
model.load_autosplit(cache)
model.cache = cache

tokenizer = ExLlamaV2Tokenizer(config)

gptq_instruct_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='exllama')
    gptq_instruct_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='exllama'
    )
    gptq_instruct_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['gptq_instruct'] = {
    'results': gptq_instruct_results,
    'backend': 'exllamav2',
    'quantization': 'GPTQ',
    'model_type': 'instruct'
}

del model, cache, tokenizer, config
clear_memory()

save_json(all_results, OUTPUT_DIR / 'gptq_instruct_results.json')

AWQ Base

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

clear_memory()

logger.info("AWQ BASE MODEL")

tokenizer = AutoTokenizer.from_pretrained('TheBloke/Mistral-7B-v0.1-AWQ')
tokenizer.pad_token = tokenizer.eos_token

model = AutoAWQForCausalLM.from_quantized(
    'TheBloke/Mistral-7B-v0.1-AWQ',
    fuse_layers=True
)
model.eval()

awq_base_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    awq_base_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    awq_base_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['awq_base'] = {
    'results': awq_base_results,
    'backend': 'autoawq',
    'quantization': 'AWQ',
    'model_type': 'base'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'awq_base_results.json')

AWQ Instruct

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

clear_memory()

logger.info("AWQ INSTRUCT MODEL")

tokenizer = AutoTokenizer.from_pretrained('TheBloke/Mistral-7B-Instruct-v0.2-AWQ')
tokenizer.pad_token = tokenizer.eos_token

model = AutoAWQForCausalLM.from_quantized(
    'TheBloke/Mistral-7B-Instruct-v0.2-AWQ',
    fuse_layers=True
)
model.eval()

awq_instruct_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    awq_instruct_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    awq_instruct_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['awq_instruct'] = {
    'results': awq_instruct_results,
    'backend': 'autoawq',
    'quantization': 'AWQ',
    'model_type': 'instruct'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'awq_instruct_results.json')

NF4 Base

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

clear_memory()

logger.info("NF4 BASE MODEL")

tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1')
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-v0.1',
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

nf4_base_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    nf4_base_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    nf4_base_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['nf4_base'] = {
    'results': nf4_base_results,
    'backend': 'bitsandbytes',
    'quantization': 'NF4',
    'model_type': 'base'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'nf4_base_results.json')

NF4 Instruct

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

clear_memory()

logger.info("NF4 INSTRUCT MODEL")

tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2',
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

nf4_instruct_results = {}

logger.info("\nPart 1: Baseline Perplexity")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating {ds_name}")
    results = evaluate_perplexity(model, tokenizer, dataset, ds_name, NUM_SAMPLES, MAX_LENGTH, backend='huggingface')
    nf4_instruct_results[ds_name] = results
    
    if 'error' not in results:
        logger.info(f"  Perplexity: {results['perplexity']:.2f}")
        logger.info(f"  Bits/Byte: {results['bits_per_byte']:.4f}")
        dist = results['distribution_analysis']
        if 'central_tendency' in dist:
            ci = dist['central_tendency']['ci_95']
            logger.info(f"  95% CI: [{ci['lower']:.2f}, {ci['upper']:.2f}]")

logger.info("\nPart 2: Context Impact")
for ds_name, dataset in loaded_datasets.items():
    logger.info(f"\nEvaluating context impact on {ds_name}")
    context_results = evaluate_context_impact(
        model, tokenizer, dataset, ds_name, CONTEXT_SAMPLES, MIN_DOC_LENGTH, backend='huggingface'
    )
    nf4_instruct_results[f'{ds_name}_context_impact'] = context_results
    
    if 'error' not in context_results:
        logger.info(f"  Without context: {context_results['without_context']['mean']:.2f}")
        logger.info(f"  With context: {context_results['with_context']['mean']:.2f}")
        logger.info(f"  Reduction: {context_results['reduction']['percent']:.1f}%")
        logger.info(f"  p-value: {context_results['significance']['p_value']:.4f}")
        logger.info(f"  Effect size: {context_results['significance']['effect_size']:.3f}")

all_results['nf4_instruct'] = {
    'results': nf4_instruct_results,
    'backend': 'bitsandbytes',
    'quantization': 'NF4',
    'model_type': 'instruct'
}

del model, tokenizer
clear_memory()

save_json(all_results, OUTPUT_DIR / 'nf4_instruct_results.json')

Final Analysis and Summary

In [ ]:
save_json(all_results, OUTPUT_DIR / 'all_results_final.json')

logger.info("\nPERPLEXITY AND CONTEXT IMPACT EVALUATION COMPLETE")
logger.info(f"\nTotal configurations: {len(all_results)}")
logger.info(f"Datasets: {list(loaded_datasets.keys())}")
logger.info(f"Baseline samples per dataset: {NUM_SAMPLES}")
logger.info(f"Context impact samples: {CONTEXT_SAMPLES}")

logger.info("BASELINE PERPLEXITY SUMMARY")

for config_name, config_data in all_results.items():
    logger.info(f"\n{config_name.upper()}")
    results_dict = config_data.get('results', {})
    
    for ds_name in loaded_datasets.keys():
        if ds_name in results_dict and 'error' not in results_dict[ds_name]:
            ppl = results_dict[ds_name]['perplexity']
            bpb = results_dict[ds_name]['bits_per_byte']
            logger.info(f"  {ds_name}: PPL={ppl:.2f}, BPB={bpb:.4f}")

logger.info("CONTEXT IMPACT SUMMARY")

for config_name, config_data in all_results.items():
    results_dict = config_data.get('results', {})
    
    has_context = any('context_impact' in key for key in results_dict.keys())
    
    if has_context:
        logger.info(f"\n{config_name.upper()}")
        for key, ctx_data in results_dict.items():
            if 'context_impact' in key and 'error' not in ctx_data:
                ds_name = key.replace('_context_impact', '')
                logger.info(f"  {ds_name}:")
                logger.info(f"    Without context: {ctx_data['without_context']['mean']:.2f}")
                logger.info(f"    With context: {ctx_data['with_context']['mean']:.2f}")
                logger.info(f"    Reduction: {ctx_data['reduction']['percent']:.1f}%")
                logger.info(f"    p-value: {ctx_data['significance']['p_value']:.4f}")
                logger.info(f"    Significant: {ctx_data['significance']['significant']}")

logger.info(f"\nResults saved to: {OUTPUT_DIR}")
logger.info("Evaluation complete")